In [1]:
""" Implementation of the nmf example using alternating projections instead of douglas-rachford."""
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF
import jax
from jax import numpy as jnp


In [2]:
""" Bilear projection using Newton's method. (from pjaxs library) """
newtonSteps = 10
def pos(m):
    m[m<0] = 0
    return m

def bilinear_proj(a, b, z, /):
    """Project onto bilinear function graph using Newton's method."""
    p = a @ b
    q = a @ a + b @ b
    def f(t):
        return ((1 + t**2) * p + t * q) / (1 - t**2) ** 2 - z + t
 
    f_prime = jax.grad(f)

    def newton_step(t):
        return t - f(t) / f_prime(t)

    t = jax.lax.fori_loop(0, newtonSteps, lambda _, t: newton_step(t), 0.0, unroll=False)

    a_new = (a + t * b) / (1 - t**2)
    b_new = (b + t * a) / (1 - t**2)

    return a_new, b_new


In [3]:
### nmf using alternating projections   
samples=3

dimension=3
rankW = 3
Y = np.random.rand(dimension, samples)
W_A = np.random.rand(dimension, rankW )
X_A = np.random.rand(rankW, samples )
X_A = np.maximum(X_A, 0)
W_i = np.zeros((samples,dimension, rankW))
X_i = np.zeros((samples, rankW, samples))

### perform nmf
norm = []
for iter in range(100):
    norm.append(np.linalg.norm( W_A @ X_A - Y))
    for k in range(samples):
        x = np.zeros_like(X_A)
        w = np.zeros_like(W_A)

        ## projection of w on non-negative orthant
        w = pos(W_A)
        ## projection of x on non-negative orthant
        x[:,k] =  np.maximum(X_A[:,k],0)      
        tempx = np.zeros((rankW, rankW))
        for i in range(rankW):
            w[i,:],tempx[:,i] = bilinear_proj(w[i,:],x[:,k], Y[i,k])
        x = np.mean(tempx, axis=1)
        W_i[k,:,:] = w
        X_i[k,:,:] = x
    W_A = np.mean(W_i, axis=0)
    X_A = np.mean(X_i, axis=1)
                
    
import matplotlib.pyplot as plt
plt.plot(norm)
model = NMF(n_components=3, init='random', random_state=0)
W = model.fit_transform(Y)
H = model.components_
print(min(norm), np.argmin(norm))
print(np.linalg.norm(W @ H - Y))
print(W_A )
print(X_A)

KeyboardInterrupt: 

In [6]:
### nmf using douglas -rachford 
samples=3

dimension=3
rankW = 3
Y = np.random.rand(dimension, samples)
W_A = np.random.rand(dimension, rankW )
X_A = np.random.rand(rankW, samples )
X_A = np.maximum(X_A, 0)
W_prev = np.zeros((dimension, rankW))
X_prev = np.zeros( (rankW, samples))

### perform nmf
norm = []
for iter in range(50):
    norm.append(np.linalg.norm( W_A @ X_A - Y))
    for k in range(samples):
        x = np.zeros_like(X_A[:k])
        w = np.zeros_like(W_A)

        ## reflection of w on non-negative orthant
        w = 2*pos(W_A) - W_A
        ## reflection of x on non-negative orthant
        x =  2*np.maximum(X_A[:,k],0) - X_A[:,k]
        tempx = np.zeros((rankW, rankW))
        
        for i in range(rankW):
            print(w[i,:].shape,x.shape, Y[i,k].shape)
            pwi,tempx[:,i] = bilinear_proj(w[i,:],x, Y[i,k])
            w[i,:] = 2*pwi - w[i,:]
            tempx[:,i] = 2*tempx[:,i] - x

        x = np.mean(tempx, axis=1)
        w_new = 0.5*(w + W_A)
        x_new = 0.5*(x + X_A[:,k])
        
        W_A = W_prev + (1/(k+1))*(w_new - W_prev)
        X_A[:,k] = X_prev[:,k] + (1/(k+1))*(x_new - X_prev[:,k])
        
        W_prev = W_A
        X_prev[:,k] = X_A[:,k]
        
import matplotlib.pyplot as plt
plt.plot(norm)
model = NMF(n_components=3, init='random', random_state=0)
W = model.fit_transform(Y)
H = model.components_
print(min(norm), np.argmin(norm))
print(np.linalg.norm(W @ H - Y))
print(W_A )
print(X_A)

(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()
(3,) (3,) ()

KeyboardInterrupt: 

In [6]:
ITERS = 1000

In [ ]:
### 2 layer neural network
### input layer 2 neurons hidden layer 2 neurons output layer 1 neuron
### approximate the xor function

# XOR dataset (inputs in R^2, labels in {0,1})
xor_X = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]], dtype=np.float32)
xor_y = np.array([0, 1, 1, 0], dtype=np.float32).reshape(-1, 1)

print("xor_X:\n", xor_X)
print("xor_y:\n", xor_y)
### define weight matrices
W0 = np.random.rand(2,2)
W1 = np.random.rand(2,1)
bias = np.random.rand(1)
delta = 0.5
def relu(x):
    return np.maximum(0,x)
### perform optimization
error = []
for iter in range(ITERS):
    prev_bias = np.copy(bias)
    err = []
    for k in range(len(xor_X)):
        w0 = np.copy(W0)
        w1 = np.copy(W1)
        ### apply A constraints
        x_in = np.copy(xor_X[k,:]).reshape(2,1)
        x_middle = W0.T @ x_in
        y_middle = relu(x_middle)
        x_out = W1.T @ y_middle -bias
        y_out = relu(x_out)
        err.append((y_out - xor_y[k])**2)

        ### set difference between code and non code outputs
        if xor_y[k] == 1:
            bias = 2*np.minimum(bias, y_out) -bias
        else:
            bias = 2*np.maximum(bias, y_out) -bias
    
        ### apply B constraints
        ##consenus for output layer   
        projection_w1,projection_x_middle = bilinear_proj(w1.reshape(2),x_middle.reshape(2), (xor_y[k] +bias)[0][0])
        reflection_w1 = 2*projection_w1 - w1.reshape(2)
        
        reflection_x_middle = 2*projection_x_middle - x_middle.reshape(2)
        
        
        ## consensus for hidden layer
        x_in_ref = np.zeros((2,2))
        projection_w0 = np.zeros_like(w0)
        reflection_w0 = np.zeros_like(w0)
        for i in range(2):
            projection_w0[:,i],projection_x_in = bilinear_proj(w0[:,i].reshape(2),x_in.reshape(2), relu(projection_x_middle)[i])
            reflection_w0[:,i] = 2*projection_w0[:,i] - w0[:,i].reshape(2)
            reflection_x_in = 2*projection_x_in - x_in.reshape(2)
            
            x_in_ref[i,:] = reflection_x_in
        
        x_in_ref = np.mean(x_in_ref, axis=0)
        #print("reflection wo",reflection_w0)
        w0_new = 0.5*(reflection_w0 + W0)

        w1_new = 0.5*(reflection_w1 + W1.reshape(2))
        
        x_middle_new = 0.5*(reflection_x_middle + x_middle)
        W0 = W0 + (1/(k+1))*(w0_new - W0)
        #print("w0_new:", w0_new)
        #print("W0:", W0)

        W1 = W1.reshape(2) + (1/(k+1))*(w1_new - W1.reshape(2))
        bias = prev_bias + (1/(k+1))*(bias - prev_bias)
    error.append(np.mean(err))
    print(error[-3:    ])

import matplotlib.pyplot as plt
plt.plot(error)


xor_X:
 [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
xor_y:
 [[0.]
 [1.]
 [1.]
 [0.]]
[np.float64(0.4249054305255413)]
[np.float64(0.4249054305255413), np.float32(0.43422452)]
[np.float64(0.4249054305255413), np.float32(0.43422452), np.float32(0.44289672)]
[np.float32(0.43422452), np.float32(0.44289672), np.float32(0.4428301)]
[np.float32(0.44289672), np.float32(0.4428301), np.float32(0.44054556)]
[np.float32(0.4428301), np.float32(0.44054556), np.float32(0.44104168)]
[np.float32(0.44054556), np.float32(0.44104168), np.float32(0.44654197)]
[np.float32(0.44104168), np.float32(0.44654197), np.float32(0.4567595)]
[np.float32(0.44654197), np.float32(0.4567595), np.float32(0.46974245)]
[np.float32(0.4567595), np.float32(0.46974245), np.float32(0.4829972)]
[np.float32(0.46974245), np.float32(0.4829972), np.float32(0.49441916)]
[np.float32(0.4829972), np.float32(0.49441916), np.float32(0.50277805)]
[np.float32(0.49441916), np.float32(0.50277805), np.float32(0.507747)]
[np.float32(0.50277805), np.floa

In [ ]:
### 2 layer neural network
### input layer 2 neurons hidden layer 2 neurons output layer 1 neuron
### approximate the xor function

# XOR dataset (inputs in R^2, labels in {0,1})
xor_X = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]], dtype=np.float32)
xor_y = np.array([0, 1, 1, 0], dtype=np.float32).reshape(-1, 1)

print("xor_X:\n", xor_X)
print("xor_y:\n", xor_y)
### define weight matrices
W0 = np.random.rand(2,2)
W1 = np.random.rand(2)
bias = np.random.rand(1)
delta = 0.5
def relu(x):
    return np.maximum(0,x)
### perform optimization
error = []
for iter in range(ITERS):
    prev_bias = np.copy(bias)
    err = []
    for k in range(len(xor_X)):
        w0 = np.copy(W0)
        w1 = np.copy(W1)
        ### apply A constraints
        x_in = np.copy(xor_X[k,:]).reshape(2)
        x_middle = W0.T @ x_in
        y_middle = relu(x_middle)
        x_out = W1.T @ y_middle -bias
        y_out = relu(x_out)
        err.append((y_out - xor_y[k])**2)

        ### set difference between code and non code outputs
        if xor_y[k] == 1:
            bias = 2*np.minimum(bias, y_out) -bias
        else:
            bias = 2*np.maximum(bias, y_out) -bias
    
        ### apply B constraints
        ##consenus for output layer   
        projection_w1,projection_x_middle = bilinear_proj(w1,x_middle, (xor_y[k] +bias)[0][0])
        reflection_w1 = 2*projection_w1 - w1
        
        reflection_x_middle = 2*projection_x_middle - x_middle
    
        
        ## consensus for hidden layer
        x_in_ref = np.zeros((2,2))
        projection_w0 = np.zeros_like(w0)
        reflection_w0 = np.zeros_like(w0)
        for i in range(2):
            projection_w0[:,i],projection_x_in = bilinear_proj(w0[:,i],x_in, relu(projection_x_middle)[i])
            reflection_w0[:,i] = 2*projection_w0[:,i] - w0[:,i]
            reflection_x_in = 2*projection_x_in - x_in
            
            x_in_ref[i,:] = reflection_x_in
        
        x_in_ref = np.mean(x_in_ref, axis=0)
        #print("reflection wo",reflection_w0)
        w0_new = 0.5*(reflection_w0 + W0)

        w1_new = 0.5*(reflection_w1 + W1)
        
        x_middle_new = 0.5*(reflection_x_middle + x_middle)
        W0 = W0 + (1/(k+1))*(w0_new - W0)
        #print("w0_new:", w0_new)
        #print("W0:", W0)

        W1 = W1 + (1/(k+1))*(w1_new - W1)
        bias = prev_bias + (1/(k+1))*(bias - prev_bias)
    error.append(np.mean(err))
    print(error[-3:    ])

import matplotlib.pyplot as plt
plt.plot(error)
